# Boosted Decision Tree

In [1]:
import pandas as pd
import numpy as np

from joblib import Parallel, delayed

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.multioutput import MultiOutputClassifier

from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')
# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Lavoro su singola fold

In [2]:
def fit_single_fold(train_idx, test_idx, features, target, groups, max_iter, max_depth, min_samples_leaf, learning_rate, l2_regularization, max_leaf_nodes, early_stopping):


    # ========== DEBUGGING: Stampo indici train/test  ==========
    
    """print("?"*50 + "\nDebug\n" + "?"*50)
    print(f"\nFold {fold} - File: {csv_name}")        
    print(f"  Train indice: {train_index[:10]})")
    print(f"  Test indice: {test_index[:10]})")
    print(f"  Train gruppo (Patient IDs): {groups.iloc[train_index].unique()}")
    print(f"  Test gruppo (Patient IDs): {groups.iloc[test_index].unique()}")
    print("?"*100)"""
    # ==========================================================

    X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
    y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

    base_clf = HistGradientBoostingClassifier(
        max_iter=max_iter,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        learning_rate=learning_rate,
        l2_regularization=l2_regularization,
        max_leaf_nodes=max_leaf_nodes,
        early_stopping=early_stopping,
        random_state=42
    )

    rf = MultiOutputClassifier(base_clf, n_jobs=-1)

    # Addestro il modello sul train set di questa fold
    rf.fit(X_train, y_train)
    
    # Predico il target sul test set di questa fold
    y_pred = rf.predict(X_test)
    
    # DEBUG
    #score = f1_score(y_test, y_pred, average="micro")
    #print(f"Fold {fold} - {max_iter=}, {max_depth=}, {min_samples_leaf=}, Score={score:.4f}")

    return f1_score(y_test, y_pred, average="micro")

# Training


In [3]:
def training(file_path, csv_name):
    # Leggo i csv
    df = pd.read_csv(file_path)

    # Mi definisco la lista dei target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']

    # Filtro solo le pazienti con PR valido
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo tutto in valori binari per "facilitare" il lavoro
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    
    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']
    
    """ 
        Preparo le feature (X) e i target (y) per il modello
    """
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]
    
    # 'groups' contiene l'ID del paziente per ogni lesione.
    # Mi serve per fare la cross-validation a gruppo
    groups = df_validi['Patient ID']

    # Riempip a Nan se è rimasto vuoto
    features = features.fillna(features.mean())


    # Imposto la strategia di cross-validation.
    # GroupKFold assicura che le lesioni dello stesso paziente non vengano mai divise tra training set e test set
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
    


    # Definisco gli iperparametri
    iperparametri = {
        'learning_rate': [0.05, 0.1, 0.2],      # Tasso di apprendimento
        'max_iter': [100, 200, 300],            # numero di iterazioni boosting
        'max_depth': [5, 10, 20, None],         # profondità massima degli alberi
        'min_samples_leaf': [1, 5, 10],         # min campioni in una foglia
        'l2_regularization': [0, 0.1, 0.5],     # Penalizza pesi troppo grandi
        'max_leaf_nodes': [31, 63, None],       # Numero massimo di foglie per albero
        'early_stopping': ['auto']              # Arresto automatico
    }


    # Lista vuota per collezionare i punteggi di performance di ogni fold.
    scores = []

    # ciclo sui valori massimi delle iterazioni
    for learning_rate in iperparametri['learning_rate']:
        for max_iter in iperparametri['max_iter']:
            for max_depth in iperparametri['max_depth']:
                for min_samples_leaf in iperparametri['min_samples_leaf']:
                    for l2_reg in iperparametri['l2_regularization']:
                        for max_leaf in iperparametri['max_leaf_nodes']:
                            for early_stop in iperparametri['early_stopping']:
                                fold_scores = Parallel(n_jobs=-1)(delayed(fit_single_fold)(train_idx, test_idx, features, target, groups, max_iter, max_depth, min_samples_leaf, learning_rate, l2_reg, max_leaf, early_stop) for train_idx, test_idx in cv.split(features, target, groups))
                                # calcolo media e deviazione standard degli score su tutte le fold
                                mean_score = np.mean(fold_scores)
                                std_score = np.std(fold_scores)

                                # registro i risultati per la combinazione di parametri corrente
                                scores.append({
                                    'max_iter': max_iter,
                                    'max_depth': max_depth,
                                    'min_samples_leaf': min_samples_leaf,
                                    'mean_score': mean_score,
                                    'std_score': std_score,
                                    'fold_scores': fold_scores
                                })


                        
    return scores

# Lettura dei file

In [4]:
results_per_dataset = {}

for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Stampo i risultati per ogni dataset
for name, metrics_list in results_per_dataset.items():
    
    # Stampo solo i migliori
    best_result = max(metrics_list, key=lambda x: x['mean_score'])
    print(f"\nDataset: {name}:")
    print(f"  max_iter: {best_result['max_iter']}")
    print(f"  max_depth: {best_result['max_depth']}")
    print(f"  min_samples_leaf: {best_result['min_samples_leaf']}")
    print(f"  Mean F1-score: {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")



# va dentro al for sopra


KeyboardInterrupt: 